# 🏆 Amazon ML Challenge 2026: Master Entity Resolution Pipeline

### Business Entity Resolution across 3 Noisy Sources optimizing Macro F0.5
This master notebook executes the complete end-to-end competition workflow:
1. **Environment Setup & Configuration** (`SAMPLE_MODE = False` for full 12.5M run on SageMaker)
2. **Exploratory Data Analysis & Country Distribution**
3. **Multi-View Normalization** (Legal suffix stripping, accent folding, address number extraction)
4. **Country-Partitioned Multi-Pass Candidate Blocking** (Measures candidate recall & reduction ratio)
5. **Pairwise Feature Engineering & S2-S3 Cross-Source Consensus**
6. **5-Fold GroupKFold LightGBM Classifier Training**
7. **Out-of-Fold Macro F0.5 Decision Policy Optimization** (`tau_match`, `tau_singleton`, `score_margin`)
8. **Full Test Inference & Output Generation** (`matching_results.tsv` and `candidate_pairs.tsv`)
9. **Official Submission Validation** (`resources/student_resource/utils/validate_submission.py`)


In [ ]:
# Section 1: Environment Setup & Project Path Configuration
import os
import sys
from pathlib import Path

# Ensure project root is on sys.path
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT}")
import numpy as np
import pandas as pd
import yaml
from rich import print as rprint


In [ ]:
# Section 2: Global Configuration & Execution Controls
# Toggle SAMPLE_MODE = False when executing full 12.5M train / 11.7M test run on SageMaker GPU/CPU instance
SAMPLE_MODE = False           # Set to True for fast local debug, False for official run
SAMPLE_N_S1 = 5000            # S1 count when running in sample mode

TRAIN_DIR = str(PROJECT_ROOT / "resources" / "student_resource" / "dataset" / "train")
TEST_DIR = str(PROJECT_ROOT / "resources" / "student_resource" / "dataset" / "test")
OUTPUT_DIR = str(PROJECT_ROOT / "output")

print(f"Execution Mode: {'SUBSAMPLE BENCHMARK' if SAMPLE_MODE else 'FULL INDUSTRIAL SCALE'}")
print(f"Train Directory: {TRAIN_DIR}")
print(f"Test Directory:  {TEST_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")


In [ ]:
# Section 3: Data Ingestion & Data Profiling
from src.data.loader import load_source_tsv, load_ground_truth, load_benchmark_subset

if SAMPLE_MODE:
    print(f"Loading representative benchmark subset (N={SAMPLE_N_S1:,})...")
    s1_records, target_records, ground_truth = load_benchmark_subset(
        data_dir=TRAIN_DIR,
        n_s1=SAMPLE_N_S1,
        background_noise_ratio=25,
    )
else:
    print("Loading full training datasets...")
    s1_records = load_source_tsv(os.path.join(TRAIN_DIR, "train_source1.tsv"))
    s2_records = load_source_tsv(os.path.join(TRAIN_DIR, "train_source2.tsv"))
    s3_records = load_source_tsv(os.path.join(TRAIN_DIR, "train_source3.tsv"))
    target_records = s2_records + s3_records
    ground_truth = load_ground_truth(os.path.join(TRAIN_DIR, "train_ground_truth.tsv"))

print(f"✅ Ingestion Complete:")
print(f"   Source 1 Reference Records: {len(s1_records):,}")
print(f"   Target Pool Records (S2+S3): {len(target_records):,}")
print(f"   Ground Truth Label Count:   {len(ground_truth):,}")


In [ ]:
# Section 4: Country-Partitioned Multi-Pass Candidate Blocking
from src.blocking.blocker import MultiPassBlocker, evaluate_blocking, export_candidate_pairs_tsv

print("Initializing Country-Partitioned Multi-Pass Blocker...")
blocker = MultiPassBlocker(
    max_candidates_per_entity=50,
    max_bucket_size=150,
    enable_name_exact=True,
    enable_name_legal_stripped=True,
    enable_name_first_two=True,
    enable_addr_num_token=True,
    enable_addr_pair_tokens=True,
    enable_name_prefix_num=True,
)

print(f"Indexing {len(target_records):,} target records...")
blocker.fit_targets(target_records)

print(f"Querying candidates for {len(s1_records):,} reference records...")
candidates_dict = blocker.block_all(s1_records)

blocking_metrics = evaluate_blocking(candidates_dict, ground_truth, len(target_records))
print("\n--- Blocking Quality Report ---")
print(f"Candidate Recall Ceiling: {blocking_metrics['candidate_recall']*100:.2f}%")
print(f"Average Candidates / S1:  {blocking_metrics['avg_candidates_per_s1']:.2f}")
print(f"P95 Candidates / S1:      {blocking_metrics['p95_candidates_per_s1']:.2f}")
print(f"Reduction Ratio:          {blocking_metrics['reduction_ratio']*100:.4f}%")
print(f"Total Candidate Pairs:    {blocking_metrics['total_candidate_pairs']:,}")


In [ ]:
# Section 5: Pairwise Feature Generation & S2-S3 Cross-Source Consensus
from src.features.pairwise_features import build_pairwise_feature_matrix, precompute_record_views
from src.features.consensus import compute_s2_s3_consensus

s1_dict = {r["entity_id"]: r for r in s1_records}
target_dict = {r["entity_id"]: r for r in target_records}

print("Computing pairwise feature matrix...")
X, y, pairs = build_pairwise_feature_matrix(
    s1_dict=s1_dict,
    target_dict=target_dict,
    candidates_dict=candidates_dict,
    ground_truth=ground_truth,
)

print("Computing cross-source S2-S3 consensus features...")
active_target_ids = {tid for _, tid in pairs}
target_precomputed = {
    tid: precompute_record_views(target_dict[tid])
    for tid in active_target_ids
    if tid in target_dict
}
consensus_df = compute_s2_s3_consensus(pairs, target_dict, target_precomputed)
X = pd.concat([X, consensus_df], axis=1)

print(f"Feature Matrix Shape: {X.shape} ({len(X):,} candidate pairs, {X.shape[1]} features)")
print(f"Positive Match Ratio: {(y == 1).mean()*100:.2f}%")


In [ ]:
# Section 6: 5-Fold GroupKFold LightGBM Model Training
from src.models.classifier import LightGBMPairClassifier
from src.validation.splitters import get_entity_splits

s1_ids = list(s1_dict.keys())
entity_splits = get_entity_splits(s1_ids, n_splits=5, shuffle=True, seed=42)

oof_probs = np.zeros(len(X), dtype=float)
pair_s1_indices = {sid: [] for sid in s1_ids}
for idx, (sid, _) in enumerate(pairs):
    pair_s1_indices[sid].append(idx)

print("Training 5-Fold GroupKFold LightGBM Pair Classifier...")
fold_models = []

for fold, (train_s1_idx, val_s1_idx) in enumerate(entity_splits, 1):
    train_s1_set = {s1_ids[i] for i in train_s1_idx}
    val_s1_set = {s1_ids[i] for i in val_s1_idx}
    
    train_pair_idx = [i for sid in train_s1_set for i in pair_s1_indices[sid]]
    val_pair_idx = [i for sid in val_s1_set for i in pair_s1_indices[sid]]
    
    if not val_pair_idx:
        continue
        
    X_tr, y_tr = X.iloc[train_pair_idx], y[train_pair_idx]
    X_va, y_va = X.iloc[val_pair_idx], y[val_pair_idx]
    
    clf = LightGBMPairClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=1.0,
        early_stopping_rounds=30,
        random_state=42 + fold,
    )
    clf.fit(X_tr, y_tr, X_val=X_va, y_val=y_va)
    oof_probs[val_pair_idx] = clf.predict_proba(X_va)
    fold_models.append(clf)
    print(f"  Fold {fold} complete.")

# Final full-fit model for test inference
print("Fitting final production model on full training pairs...")
final_model = LightGBMPairClassifier(n_estimators=400, learning_rate=0.05, max_depth=6, random_state=42)
final_model.fit(X, y)
print("✅ Model training complete.")


In [ ]:
# Section 7: OOF Macro F0.5 Optimization & Decision Policy Tuning
from src.inference.decision_policy import optimize_decision_policy, PrecisionDecisionPolicy
from src.validation.metrics import compute_macro_f05

print("Searching optimal decision thresholds on OOF predictions...")
opt_policy, opt_metrics = optimize_decision_policy(
    pairs=pairs,
    val_scores=oof_probs,
    ground_truth=ground_truth,
    all_s1_ids=s1_ids,
    match_thresholds=[0.55, 0.60, 0.65, 0.70, 0.75, 0.80],
    singleton_thresholds=[0.45, 0.50, 0.55, 0.60],
    score_margins=[0.10, 0.15, 0.20, 0.25],
    verbose=False,
)

print("\n============================================================")
print(f"Optimal Match Threshold:     {opt_policy.match_threshold:.2f}")
print(f"Optimal Singleton Threshold: {opt_policy.singleton_threshold:.2f}")
print(f"Optimal Score Margin:        {opt_policy.score_margin:.2f}")
print("------------------------------------------------------------")
print(f"OOF Macro F0.5 (Official):   {opt_metrics['macro_f05']:.4f}")
print(f"OOF Macro Precision:         {opt_metrics['macro_precision']:.4f}")
print(f"OOF Macro Recall:            {opt_metrics['macro_recall']:.4f}")
print(f"OOF Singleton Accuracy:      {opt_metrics['singleton_accuracy']:.4f}")
print(f"OOF False Merges (FP):       {opt_metrics['false_merges']:,}")
print("============================================================")


In [ ]:
# Section 8: Test Inference & Candidate Export
print("Loading Test Records...")
test_s1_records = load_source_tsv(os.path.join(TEST_DIR, "test_source1.tsv"))
test_s2_records = load_source_tsv(os.path.join(TEST_DIR, "test_source2.tsv"))
test_s3_records = load_source_tsv(os.path.join(TEST_DIR, "test_source3.tsv"))
test_target_records = test_s2_records + test_s3_records

print(f"Test S1 Records: {len(test_s1_records):,}")
print(f"Test Target Records: {len(test_target_records):,}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
matching_path = os.path.join(OUTPUT_DIR, "matching_results.tsv")
candidate_path = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")

print("Running candidate blocking on test set...")
test_blocker = MultiPassBlocker(max_candidates_per_entity=50)
test_blocker.fit_targets(test_target_records)
test_candidates = test_blocker.block_all(test_s1_records)

test_s1_order = [r["entity_id"] for r in test_s1_records]
export_candidate_pairs_tsv(test_candidates, candidate_path, s1_ordered_ids=test_s1_order)
print(f"✅ Exported candidate_pairs.tsv: {candidate_path}")

print("Extracting test features...")
test_s1_dict = {r["entity_id"]: r for r in test_s1_records}
test_target_dict = {r["entity_id"]: r for r in test_target_records}

X_test, _, test_pairs = build_pairwise_feature_matrix(
    s1_dict=test_s1_dict,
    target_dict=test_target_dict,
    candidates_dict=test_candidates,
)

active_tids = {tid for _, tid in test_pairs}
tgt_precomp = {
    tid: precompute_record_views(test_target_dict[tid])
    for tid in active_tids
    if tid in test_target_dict
}
cons_test = compute_s2_s3_consensus(test_pairs, test_target_dict, tgt_precomp)
X_test = pd.concat([X_test, cons_test], axis=1)

print(f"Scoring {len(X_test):,} candidate pairs...")
test_scores = final_model.predict_proba(X_test) if not X_test.empty else np.array([])

print("Applying precision decision policy...")
matches_dict = opt_policy.apply(test_pairs, test_scores, all_s1_ids=test_s1_order)

with open(matching_path, "w", encoding="utf-8") as f:
    f.write("source1_entity_id\tmatched_entity_ids\n")
    for sid in test_s1_order:
        mids = matches_dict.get(sid, set())
        m_str = ",".join(sorted(mids)) if mids else ""
        f.write(f"{sid}\t{m_str}\n")

print(f"✅ Exported matching_results.tsv: {matching_path}")


In [ ]:
# Section 9: Official Submission Validation
from src.inference.submission_validator import validate_submission_package

print("Running official submission validation suite...")
report = validate_submission_package(
    matching_path=matching_path,
    candidate_path=candidate_path,
    test_dir=TEST_DIR,
    run_official_script=True,
)

print(report.summary())
if report.is_valid:
    print("\n🎉 SUBMISSION PACKAGE IS 100% VALIDATED AND READY FOR LEADERBOARD UPLOAD!")
else:
    raise RuntimeError(f"Validation issues found: {report.errors}")
